# Domain model example: Entity, relations, Lifecycle, Resource, Action.

Demonstrates:
  - BaseEntity with fields and lifecycle
  - AssociationOne/Many with Inverse (bidirectional, checked at startup)
  - CompositeMany — order lines owned by an order
  - OrderResource as transport layer (pure transport, no business logic)
  - CreateOrderAction using the resource via @connection — logic separate from transport

Tutorial: ../../docs/tutorials/step-21-resource.md  ·  topic: Domain model / Resource

In [ ]:
!pip install -q aoa-action-machine

In [ ]:
from __future__ import annotations

import asyncio
import uuid
from typing import Annotated, Any

from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain import (
    AssociationMany,
    AssociationOne,
    BaseCoreEntity,
    BaseDetailEntity,
    BaseStateEntity,
    CompositeMany,
    Inverse,
    Lifecycle,
    build,
    value,
)
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import regular_aspect, summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.checkers import result_float, result_string
from aoa.action_machine.intents.connection import connection
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.model.base_state import BaseState
from aoa.action_machine.resources import Connections
from aoa.action_machine.resources.base_resource import BaseResource
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.action_machine.runtime.tools_box import ToolsBox

# ---------------------------------------------------------------------------
# Domain
# ---------------------------------------------------------------------------


class StoreDomain(BaseDomain):
    name = "store"
    description = "E-commerce store domain"


# ---------------------------------------------------------------------------
# Lifecycles
# ---------------------------------------------------------------------------


class OrderStateEntity(BaseStateEntity):
    """Status of the order axis."""

    sort_order: int = Field(default=0, description="Presentation order")

    NEW = value(code="new", display_name="New", sort_order=1)
    CONFIRMED = value(code="confirmed", display_name="Confirmed", sort_order=2)
    SHIPPED = value(code="shipped", display_name="Shipped", sort_order=3)
    DELIVERED = value(code="delivered", display_name="Delivered", sort_order=4)
    CANCELLED = value(code="cancelled", display_name="Cancelled", sort_order=5)



class OrderLifecycle(Lifecycle[OrderStateEntity]):

    _template = (
        Lifecycle()
        .state(OrderStateEntity.NEW)
        .to(OrderStateEntity.CONFIRMED, OrderStateEntity.CANCELLED)
        .initial()
        .state(OrderStateEntity.CONFIRMED)
        .to(OrderStateEntity.SHIPPED, OrderStateEntity.CANCELLED)
        .intermediate()
        .state(OrderStateEntity.SHIPPED)
        .to(OrderStateEntity.DELIVERED)
        .intermediate()
        .state(OrderStateEntity.DELIVERED)
        .final()
        .state(OrderStateEntity.CANCELLED)
        .final()
    )


class OrderLineStateEntity(BaseStateEntity):
    """Status of the order-line axis."""

    sort_order: int = Field(default=0, description="Presentation order")

    PENDING = value(code="pending", display_name="Pending", sort_order=1)
    RESERVED = value(code="reserved", display_name="Reserved", sort_order=2)
    SHIPPED = value(code="shipped", display_name="Shipped", sort_order=3)
    DELIVERED = value(code="delivered", display_name="Delivered", sort_order=4)
    CANCELLED = value(code="cancelled", display_name="Cancelled", sort_order=5)



class OrderLineLifecycle(Lifecycle[OrderLineStateEntity]):

    _template = (
        Lifecycle()
        .state(OrderLineStateEntity.PENDING)
        .to(OrderLineStateEntity.RESERVED, OrderLineStateEntity.CANCELLED)
        .initial()
        .state(OrderLineStateEntity.RESERVED)
        .to(OrderLineStateEntity.SHIPPED)
        .intermediate()
        .state(OrderLineStateEntity.SHIPPED)
        .to(OrderLineStateEntity.DELIVERED)
        .intermediate()
        .state(OrderLineStateEntity.DELIVERED)
        .final()
        .state(OrderLineStateEntity.CANCELLED)
        .final()
    )


# ---------------------------------------------------------------------------
# Entities
# ---------------------------------------------------------------------------

# Forward-reference pattern: all three classes are declared in one module
# so Inverse() can reference partner types at class-body time.


@entity(description="Registered customer account", domain=StoreDomain)
class CustomerCoreEntity(BaseCoreEntity):
    id: str = Field(description="Customer identifier")
    name: str = Field(description="Display name")
    email: str = Field(description="Contact email")

    # association many — customer exists independently of any order
    orders: Annotated[
        AssociationMany[OrderCoreEntity] | None,
        Inverse(OrderCoreEntity, "customer"),
    ] = Field(default=None, description="Orders placed by this customer; NULL when not loaded")


@entity(description="Customer order", domain=StoreDomain)
class OrderCoreEntity(BaseCoreEntity):
    id: str = Field(description="Order identifier")
    total: float = Field(description="Order total", ge=0)
    currency: str = Field(description="Currency code", default="RUB")
    lifecycle: OrderLifecycle = Field(description="Order lifecycle state")

    # association one-to-one — order and customer exist independently
    customer: Annotated[
        AssociationOne[CustomerCoreEntity],
        Inverse(CustomerCoreEntity, "orders"),
    ] = Field(description="Customer who placed the order")

    # composition one-to-many — order lines cannot exist without this order
    lines: Annotated[
        CompositeMany[OrderLineDetailEntity],
        Inverse(OrderLineDetailEntity, "order"),
    ] = Field(description="Line items of the order")


@entity(description="Single line item within an order", domain=StoreDomain)
class OrderLineDetailEntity(BaseDetailEntity):
    id: str = Field(description="Line item identifier")
    product_id: str = Field(description="Product identifier")
    product_name: str = Field(description="Product display name")
    quantity: int = Field(description="Quantity ordered", ge=1)
    unit_price: float = Field(description="Unit price at the time of order", ge=0)
    lifecycle: OrderLineLifecycle = Field(description="Line item lifecycle state")

    # association back to parent order
    order: Annotated[
        AssociationOne[OrderCoreEntity],
        Inverse(OrderCoreEntity, "lines"),
    ] = Field(description="Parent order")


# Rebuild forward references after all three classes are defined
CustomerCoreEntity.model_rebuild()
OrderCoreEntity.model_rebuild()
OrderLineDetailEntity.model_rebuild()


# ---------------------------------------------------------------------------
# Resource — transport layer, no business logic
# ---------------------------------------------------------------------------


@meta(description="In-memory order store (transport only)", domain=StoreDomain)
class OrderResource(BaseResource):
    """In-memory stub; a production version would wrap a SQL/Postgres resource."""

    def __init__(self) -> None:
        self._orders: dict[str, dict] = {}
        self._customers: dict[str, dict] = {
            "cust-001": {"id": "cust-001", "name": "Alice", "email": "alice@example.com"},
        }

    def get_wrapper_class(self) -> type[BaseResource] | None:
        return None  # in-memory stub: direct pass-through, no transaction wrapper

    async def load_customer(self, customer_id: str) -> CustomerCoreEntity:
        row = self._customers.get(customer_id)
        if row is None:
            raise KeyError(f"Customer {customer_id!r} not found")
        return build(row, CustomerCoreEntity)

    async def save_order(self, order_id: str, customer_id: str, total: float, currency: str) -> None:
        self._orders[order_id] = {
            "id": order_id,
            "customer_id": customer_id,
            "total": total,
            "currency": currency,
            "status": "new",
        }


# ---------------------------------------------------------------------------
# Params & Result
# ---------------------------------------------------------------------------


class CreateOrderParams(BaseParams):
    customer_id: str = Field(description="Customer placing the order")
    lines: list[dict] = Field(description="Line items: [{product_id, name, qty, price}]")
    currency: str = Field(description="Currency code", default="RUB")


class CreateOrderResult(BaseResult):
    order_id: str = Field(description="Created order identifier")
    total: float = Field(description="Order total")
    customer_name: str = Field(description="Customer display name")


# ---------------------------------------------------------------------------
# Action — business logic only; transport reached through @connection
# ---------------------------------------------------------------------------


@meta(description="Create a new customer order", domain=StoreDomain)
@check_roles(GuestRole)
@connection(OrderResource, key="orders")
class CreateOrderAction(BaseAction[CreateOrderParams, CreateOrderResult]):

    # Each step forwards the fields downstream needs (state is per-step, not cumulative).

    @regular_aspect("Load and validate customer")
    @result_string("customer_name", required=True)
    async def load_customer_aspect(
        self, params: CreateOrderParams, state: BaseState, box: ToolsBox, connections: Connections
    ) -> dict[str, Any]:
        repo = connections["orders"]
        customer = await repo.load_customer(params.customer_id)
        return {"customer_name": customer.name}

    @regular_aspect("Calculate order total")
    @result_string("customer_name", required=True)
    @result_float("total", required=True, min_value=0)
    async def calculate_total_aspect(
        self, params: CreateOrderParams, state: BaseState, box: ToolsBox, connections: Connections
    ) -> dict[str, Any]:
        return {
            "customer_name": state["customer_name"],
            "total": sum(item["qty"] * item["price"] for item in params.lines),
        }

    @regular_aspect("Persist order")
    @result_string("customer_name", required=True)
    @result_float("total", required=True, min_value=0)
    @result_string("order_id", required=True)
    async def persist_order_aspect(
        self, params: CreateOrderParams, state: BaseState, box: ToolsBox, connections: Connections
    ) -> dict[str, Any]:
        repo = connections["orders"]
        order_id = f"ord-{uuid.uuid4().hex[:8]}"
        await repo.save_order(
            order_id=order_id,
            customer_id=params.customer_id,
            total=state["total"],
            currency=params.currency,
        )
        return {
            "customer_name": state["customer_name"],
            "total": state["total"],
            "order_id": order_id,
        }

    @summary_aspect("Build result")
    async def final_summary(
        self, params: CreateOrderParams, state: BaseState, box: ToolsBox, connections: Connections
    ) -> CreateOrderResult:
        return CreateOrderResult(
            order_id=state["order_id"],
            total=state["total"],
            customer_name=state["customer_name"],
        )


# ---------------------------------------------------------------------------
# Runner
# ---------------------------------------------------------------------------


async def main() -> None:
    order_resource = OrderResource()
    machine = ActionProductMachine()

    result = await machine.run(
        Context(),
        CreateOrderAction(),
        CreateOrderParams(
            customer_id="cust-001",
            lines=[
                {"product_id": "sku-1", "name": "Wireless Headphones", "qty": 1, "price": 8990.0},
                {"product_id": "sku-2", "name": "USB-C Cable", "qty": 2, "price": 490.0},
            ],
        ),
        connections={"orders": order_resource},
    )

    print(f"order_id:      {result.order_id}")
    print(f"total:         {result.total} RUB")
    print(f"customer_name: {result.customer_name}")


await main()